In [63]:
import pandas as pd 
import re

df = pd.read_csv(r'inventory_export.csv')
pd.set_option('display.max_rows', 1000)



# --- CONFIG ---
bad_folders = ["Unknown", "#recycle", "Unorganized"]   # <-- change these
suffix_pattern = r"(\(\d+\)|\s\d+\.|_\d+)$"             # matches (1), " 1.", _1, etc.

# --- SCORING FUNCTION ---
def score_row(row):
    score = 100  # start neutral / good
    
    folder = row['filepath'].lower()

    # Rule 1: If folder path contains bad strings → penalize
    if any(bad in folder for bad in bad_folders):
        score -= 50

    # Rule 2: If filename has a duplicate-style suffix → penalize
    filename = row['filename']
    if re.search(suffix_pattern, filename):
        score -= 100  # strong penalty

    return score

# 1. detect duplicate groups
df['dupe_group'] = df.groupby('sha256')['sha256'].transform('count') > 1

# 2. scoring from before
df['score'] = df.apply(score_row, axis=1)

# 3. tie-breaker so only one row can ever be max
df['tie_breaker'] = -df['filepath'].str.len()   # shorter path wins
df['final_score'] = df['score'] * 1_000_000 + df['tie_breaker']

# 4. compute group max
df['max_final'] = df.groupby('sha256')['final_score'].transform('max')

# 5. keep the single highest scorer ONLY within dupe groups
df['keep'] = True
df.loc[df['dupe_group'], 'keep'] = (
    df.loc[df['dupe_group'], 'final_score'] == df.loc[df['dupe_group'], 'max_final']
)

# 6. everything else in duplicate groups is marked duplicate
df['isDuplicate'] = ~df['keep']



In [78]:
media_ext = (".mov", ".mp4", ".wav", ".m4v", ".avi", ".flac", ".mp3", ".mkv")
pd.set_option('display.max_rows', None)
dff = df[(df['keep'] == True ) & (df["filename"].str.lower().str.endswith(media_ext))][['filepath', 'keep', 'filesize']].sort_values(by='filepath', ascending=False)




In [79]:
dff.to_csv(r'20250206_inventory_to_migrate.csv')


In [77]:
import os
os.getcwd()



'/Users/auston/Repos/digital-asset-manager'